# FRIDGE RESCUE — End-to-End ML Pipeline
**Tagline**: *Rescue food before it becomes waste.*

### Alignment
- **SDG 12**: Responsible Consumption and Production (Target 12.3: Halve household food waste)
- **SDG 2**: Zero Hunger

### Pipeline Overview
1. **Dataset Download & Preprocessing** (Northumbria Smart Refrigerator + Kaggle Fresh/Stale)
2. **Dataset Integrity Verification**
3. **Food Object Detection Model** (YOLO11n fine-tuning)
4. **Produce Freshness Classifier** (MobileNetV3 transfer learning)
5. **OCR Packaging Expiry Extractor** (EasyOCR + ISO Normalization)
6. **Multi-Signal Rescue Priority Decision Engine**
7. **Rescue Action Agent** (Culinary waste-reduction actions)
8. **Full End-to-End Refrigerator Scan Demo**

In [ ]:
# Step 1: Environment & Imports
import os
import sys
from pathlib import Path
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from dotenv import load_dotenv

# Set project root
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# Step 2: Verify Both Mandatory Datasets Programmatically
from scripts.verify_datasets import verify_smart_refrigerator, verify_fresh_stale

ok1, samples1 = verify_smart_refrigerator()
ok2, samples2 = verify_fresh_stale()
print(f"Smart Refrigerator Dataset OK: {ok1}")
print(f"Fresh & Stale Dataset OK: {ok2}")

In [ ]:
# Step 3: Inspect Mandatory Freshness Dataset Evaluation Results
import json
metrics_path = PROJECT_ROOT / "ml" / "freshness" / "evaluation" / "metrics.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        f_metrics = json.load(f)
    print("Freshness Classifier Test Metrics:")
    print(f"  Accuracy:  {f_metrics.get('accuracy', 0)*100:.2f}%")
    print(f"  Precision: {f_metrics.get('precision', 0)*100:.2f}%")
    print(f"  Recall:    {f_metrics.get('recall', 0)*100:.2f}%")
    print(f"  F1 Score:  {f_metrics.get('f1_score', 0)*100:.2f}%")
    print(f"  Confusion Matrix: {f_metrics.get('confusion_matrix')}")

In [ ]:
# Step 4: Run End-to-End Pipeline on a Sample Refrigerator Photo
from ml.detection.predict import FoodDetector
from ml.freshness.predict import FreshnessClassifier
from ml.ocr.expiry_detector import ExpiryDateDetector
from ml.priority.rescue_engine import RescuePriorityEngine
from ml.action_agent.rescue_agent import RescueActionAgent

detector = FoodDetector()
freshness_clf = FreshnessClassifier()
ocr = ExpiryDateDetector()
agent = RescueActionAgent()

sample_img_path = PROJECT_ROOT / "demo" / "sample_fridge_1.jpg"
detections, annotated = detector.detect(sample_img_path)
print(f"Detected {len(detections)} food items in sample fridge image.")

In [ ]:
# Step 5: Multi-Signal Priority Ranking & Action Generation
# Group items and assess freshness / expiry
raw_items = []
for i, d in enumerate(detections[:6]):
    name = d["name"]
    crop = d["crop"]
    f_res = freshness_clf.predict_crop(crop, name)
    raw_items.append({
        "id": f"demo_item_{i+1}",
        "name": name,
        "quantity": 1,
        "detection_confidence": d["confidence"],
        "freshness": f_res.get("freshness"),
        "freshness_confidence": f_res.get("freshness_confidence"),
        "expiry_date": None
    })

ranked = RescuePriorityEngine.rank_fridge_inventory(raw_items)
rescue_plan = agent.formulate_rescue_plan(ranked)

print("\n--- RESCUE PRIORITY INVENTORY ---")
for it in ranked["items"]:
    print(f"[{it['tier']}] {it['name']} - Score: {it['score']}/100 - {it['action_tag']}")

print("\n--- ACTION AGENT PLAN ---")
print("Headline:", rescue_plan["headline"])
print("Summary:", rescue_plan["summary"])
if rescue_plan["recommended_recipes"]:
    print("Recipe:", rescue_plan["recommended_recipes"][0]["title"])